# Chapter 5：词嵌入与输出层

按顺序运行。固定六项词表、固定随机种子，CPU 即可复现；没有下载和训练完整模型。

## 1. 从 token ID 查表

先猜 `ids`、权重和输出各有几维。

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(7)
vocab = ["<pad>", "<eos>", "我", "在", "学习", "数学"]
ids = torch.tensor([[2, 3, 4], [2, 3, 5]], dtype=torch.long)
embedding = nn.Embedding(len(vocab), 4)
vectors = embedding(ids)
print("ID:", ids.shape, "词表:", embedding.weight.shape, "向量:", vectors.shape)
assert vectors.shape == (2, 3, 4)
assert torch.equal(vectors[0, 0], vectors[1, 0])

相同 ID 刚查表时得到相同向量。此时还没有位置表示或上下文计算。

In [ ]:
one_hot = F.one_hot(ids, num_classes=len(vocab)).float()
assert torch.allclose(one_hot @ embedding.weight, vectors)
print("one-hot × 参数表 = 查表:", True)

## 2. 把向量投影回词表

这里用查表结果暂代 Transformer 的隐藏状态，只验证输出端的张量形状。

In [ ]:
output = nn.Linear(4, len(vocab), bias=True)
logits = output(vectors)
probs = logits.softmax(dim=-1)
print("输出权重:", output.weight.shape, "logits:", logits.shape)
print("第一位置六个候选概率:", probs[0, 0].tolist())
assert logits.shape == (2, 3, 6)
assert torch.allclose(probs.sum(-1), torch.ones(2, 3))

## 3. 下一 token 损失与梯度

`我 在 学习 <eos>` 的输入是前三项，目标是后三项。交叉熵直接接收 logits。

In [ ]:
input_ids = torch.tensor([[2, 3, 4]])
target_ids = torch.tensor([[3, 4, 1]])
embedding.zero_grad()
output.zero_grad()
model_logits = output(embedding(input_ids))
loss = F.cross_entropy(model_logits.reshape(-1, len(vocab)), target_ids.reshape(-1))
loss.backward()
print("loss:", round(loss.item(), 4))
print("输入嵌入各行梯度范数:", embedding.weight.grad.norm(dim=1).tolist())
print("输出权重各行梯度范数:", output.weight.grad.norm(dim=1).tolist())
assert torch.count_nonzero(embedding.weight.grad[[0, 1, 5]]) == 0
assert torch.count_nonzero(output.weight.grad[5]) > 0

反向传播只写入梯度。现在用 SGD 真正更新两张表，并比较同一批数据的损失。

In [ ]:
optimizer = torch.optim.SGD(list(embedding.parameters()) + list(output.parameters()), lr=0.1)
before = loss.item()
optimizer.step()
after_logits = output(embedding(input_ids))
after = F.cross_entropy(after_logits.reshape(-1, len(vocab)), target_ids.reshape(-1)).item()
print("更新前/后:", round(before, 4), round(after, 4))
assert after < before

未作为输入的 ID 1、5 在未共享输入表中没有梯度；输出层仍为所有候选打分。`<pad>` 行在这里也没作为输入出现。

## 4. 权重共享

共享指同一个 `Parameter` 对象，不是两张表初始化成同样数字。

In [ ]:
tied_embedding = nn.Embedding(len(vocab), 4)
tied_output = nn.Linear(4, len(vocab), bias=False)
tied_output.weight = tied_embedding.weight
assert tied_output.weight is tied_embedding.weight
assert tied_output.weight.data_ptr() == tied_embedding.weight.data_ptr()
hidden = tied_embedding(input_ids)
assert torch.allclose(tied_output(hidden), F.linear(hidden, tied_embedding.weight))
print("未共享两端参数（含输出偏置）:", embedding.weight.numel() + sum(p.numel() for p in output.parameters()))
print("共享两端唯一参数:", len({id(p) for p in list(tied_embedding.parameters()) + list(tied_output.parameters())}))
print("共享权重元素数:", tied_embedding.weight.numel())

In [ ]:
tied_embedding.zero_grad()
tied_logits = tied_output(tied_embedding(input_ids))
tied_loss = F.cross_entropy(tied_logits.reshape(-1, len(vocab)), target_ids.reshape(-1))
tied_loss.backward()
print("共享表各行梯度范数:", tied_embedding.weight.grad.norm(dim=1).tolist())
assert torch.count_nonzero(tied_embedding.weight.grad[5]) > 0

ID 5 没出现在输入中，却能从输出分类路径得到梯度。共享后的输入与输出梯度汇入同一张表。

## 5. Agent 检索向量的边界

下面仅用手工特征演示“一段文本 → 一个检索向量”，不是预训练 embedding 模型，也不能代表记忆系统评测。

In [ ]:
keywords = ["Plotly", "PyTorch", "图表"]
def retrieval_vector(text):
    return torch.tensor([float(word in text) for word in keywords])
memories = ["图表统一使用 Plotly", "模型使用 PyTorch"]
query = "图表用什么"
q = retrieval_vector(query)
scored = [(F.cosine_similarity(q, retrieval_vector(m), dim=0).item(), m) for m in memories]
print(sorted(scored, reverse=True))
assert max(scored)[1] == memories[0]

检索到的文字需要重新放入提示词并经过 tokenizer；这里的检索向量不会直接塞进 `nn.Embedding`。真实记忆还需处理时间、来源、更新与删除，见 Chapter 25。

## 检查

1. 把 `ids` 改成形状 `[2, 5]`，两端的形状各是什么？
2. 为什么未共享输入表第 5 行没有梯度，共享后却有？
3. 把词表 ID 重新排序但不调整权重，会发生什么？
4. 哪个实验尚未证明模型具有语言能力？